# SADAR Finance - Behavior / Overspending Model

Notebook ini mendokumentasikan model behavior untuk mendeteksi risiko `spike` atau pola pengeluaran berlebih pada transaksi pengguna.

Model yang dibandingkan:
- **Tabular DNN / MLP** sebagai baseline deep learning.
- **Deep & Cross Network** sebagai model tabular yang menangkap interaksi fitur transaksi.

Target quest:
- TensorFlow Functional API / Model Subclassing.
- Custom Layer, Custom Loss, dan Custom Callback.
- Export `.keras` dan SavedModel.
- Inference sederhana.
- REST API Flask.
- Custom training loop dengan `tf.GradientTape`.
- TensorBoard logs.
- Accuracy minimal 85% dan MAE maksimal 0,02.

> Catatan: dataset bersifat simulasi, sehingga akurasi dapat terlihat sangat tinggi. Hasil perlu dijelaskan sebagai performa pada dataset simulasi, bukan klaim performa dunia nyata.

## 1. Setup dan Load Artefak

Notebook ini membaca dataset, metadata hasil training, dan model terbaik yang sudah diekspor dari script `ai/train_behavior.py`.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

AI_DIR = ROOT / 'ai'
sys.path.append(str(AI_DIR))

DATASET_PATH = AI_DIR / 'dataset' / 'data_modelling.csv'
METADATA_PATH = AI_DIR / 'models' / 'behavior_metadata.json'
MODEL_PATH = AI_DIR / 'models' / 'behavior_best_model.keras'

sns.set_theme(style='whitegrid')
print('Root:', ROOT)
print('Dataset exists:', DATASET_PATH.exists())
print('Metadata exists:', METADATA_PATH.exists())
print('Model exists:', MODEL_PATH.exists())

In [ ]:
df = pd.read_csv(DATASET_PATH, parse_dates=['date'])
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))

print('Dataset shape:', df.shape)
display(df.head())
metadata

## 2. EDA Singkat Dataset Behavior

Bagian ini menunjukkan distribusi target `spike`, kategori utama 50/30/20, dan pola nominal transaksi. Ini penting untuk menjelaskan mengapa model dapat mencapai performa tinggi pada dataset simulasi.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='spike', ax=axes[0], palette='Set2')
axes[0].set_title('Distribusi Label Spike')
axes[0].set_xlabel('Spike')
axes[0].set_ylabel('Jumlah Transaksi')

sns.countplot(data=df, x='category_primary', hue='spike', ax=axes[1], palette='Set2')
axes[1].set_title('Spike per Kategori 50/30/20')
axes[1].set_xlabel('Category Primary')
axes[1].set_ylabel('Jumlah Transaksi')

sns.boxplot(data=df, x='spike', y='amount', ax=axes[2], palette='Set2')
axes[2].set_title('Distribusi Nominal berdasarkan Spike')
axes[2].set_xlabel('Spike')
axes[2].set_ylabel('Amount')

plt.tight_layout()
plt.show()

In [ ]:
spike_rate = (
    df.groupby('category_detail')['spike']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='spike_rate')
)

plt.figure(figsize=(12, 5))
sns.barplot(data=spike_rate, x='category_detail', y='spike_rate', palette='viridis')
plt.title('Spike Rate per Detail Kategori')
plt.xlabel('Category Detail')
plt.ylabel('Spike Rate')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

display(spike_rate)

## 3. Hasil Perbandingan Model

Metrik diambil dari `ai/models/behavior_metadata.json`, yaitu hasil training custom loop menggunakan `tf.GradientTape`.

Model terbaik dipilih berdasarkan MAE terendah selama tetap memenuhi syarat minimum akurasi dan recall.

In [ ]:
metrics_df = pd.DataFrame(metadata['metrics']).T.reset_index().rename(columns={'index': 'model'})
display(metrics_df[['model', 'accuracy', 'mae', 'precision', 'recall', 'f1', 'tp', 'tn', 'fp', 'fn']])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=metrics_df, x='model', y='accuracy', ax=axes[0], palette='Blues')
axes[0].axhline(0.85, color='red', linestyle='--', label='Minimum Quest 85%')
axes[0].set_ylim(0.80, 1.01)
axes[0].set_title('Perbandingan Accuracy')
axes[0].legend()

sns.barplot(data=metrics_df, x='model', y='mae', ax=axes[1], palette='Greens')
axes[1].axhline(0.02, color='red', linestyle='--', label='Maksimum Quest 0,02')
axes[1].set_title('Perbandingan MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Best model:', metadata['bestModel'])

In [ ]:
best = metrics_df.loc[metrics_df['model'] == metadata['bestModel']].iloc[0]
confusion = np.array([[best['tn'], best['fp']], [best['fn'], best['tp']]], dtype=int)

plt.figure(figsize=(6, 5))
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred False', 'Pred True'], yticklabels=['Actual False', 'Actual True'])
plt.title(f"Confusion Matrix - {metadata['bestModel']}")
plt.xlabel('Prediction')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 4. Custom Component yang Digunakan

- **Custom Layer:** `CrossFeatureLayer`, digunakan oleh Deep & Cross Network untuk menangkap kombinasi fitur transaksi.
- **Custom Loss Function:** `WeightedBinaryCrossentropy`, custom BCE untuk probabilitas yang lebih terkalibrasi.
- **Custom Callback:** `QuestMetricCallback`, dipakai di custom training loop untuk memantau accuracy dan MAE sesuai syarat quest.
- **Custom Training Loop:** training dilakukan manual menggunakan `tf.GradientTape` di `ai/behavior_model.py`.

## 5. Load Model dan Inference Sederhana

Bagian ini membuktikan model `.keras` dapat diload dan dipakai untuk prediksi transaksi baru.

In [ ]:
from inference.behavior import predict_behavior

samples = [
    {
        'amount': 45000,
        'date': '2024-12-29T09:00:00',
        'merchant': 'Indomaret',
        'categoryDetail': 'groceries',
        'categoryPrimary': 'Needs',
        'paymentMethod': 'QRIS',
        'paymentMedia': 'Gopay',
    },
    {
        'amount': 1800000,
        'date': '2024-12-29T20:00:00',
        'merchant': 'Tokopedia',
        'categoryDetail': 'shopping',
        'categoryPrimary': 'Wants',
        'paymentMethod': 'Credit Card',
        'paymentMedia': 'BCA',
    },
]

predictions = [predict_behavior(sample) for sample in samples]
display(pd.DataFrame(predictions))

## 6. Integrasi Sistem

Endpoint Flask yang tersedia:

```txt
POST /behavior/predict
```

Output utama:
- `spikeProbability`
- `predictedSpike`
- `riskLevel`
- `modelName`
- `modelVersion`
- `budgetBucket`
- `recommendation`

Output model dapat digabung dengan aturan 50/30/20:
- Needs: 50%
- Wants: 30%
- Investment/Savings: 20%

## 7. Kesimpulan

Model behavior berhasil memenuhi main quest dan side quest:

- Deep learning TensorFlow Functional API: terpenuhi.
- Dua algoritma pembanding: MLP dan Deep & Cross Network.
- Custom layer/loss/callback: terpenuhi.
- Custom training loop `tf.GradientTape`: terpenuhi.
- Export `.keras` dan SavedModel: terpenuhi.
- Inference sederhana dan REST API Flask: terpenuhi.
- TensorBoard logs: tersedia di `ai/logs/behavior_spike/`.
- Accuracy >= 85% dan MAE <= 0,02: terpenuhi.

Untuk laporan akhir, gunakan narasi bahwa target performa terpenuhi, namun dataset simulasi membuat pola label sangat konsisten sehingga performa dunia nyata tetap membutuhkan validasi data aktual.